# T55 — North American thermochronology on paleo-Earth: continent-scale Phanerozoic exhumation vs Young 2022 dynamic topography

**Cluster K: Thermochronology + ThermoPlates (extended entry).**

*Based on:* Hillenbrand, I.W. et al. (2023, rev. 2025), *USGS Geochron: A Database of Geochronological and Thermochronological Dates and Data (ver. 4.0)*. U.S. Geological Survey data release, DOI: 10.5066/P9RZNPIF (public domain).

## What this notebook does

Extends the T47 → T48 → T49 → T50 ThermoPlates workflow from Central Asia to **continent-scale North America** — a Cordillera-margin + cratonic-interior system whose Mesozoic-Cenozoic uplift history is strongly coupled to the **Farallon slab dynamic-topography signal** (Moucha et al. 2008, Forte et al. 2010, Karlstrom et al. 2012). Unlike Afro-Arabia (T51 in an earlier suite), where dynamic topography was a *subordinate* driver relative to plume-driven uplift and rift-flank tectonics, dynamic topography is the **primary** geodynamic hypothesis for North American uplift, and the Young et al. (2022) `gld428` model — which explicitly captures subduction-coupled DT — is well-suited to represent it.

**Sections:**
1. **§1** Load plate model + USGS Geochron v4.0 North American thermochronology compilation.
2. **§2** Per-sample cooling rate at the snapshot age (closure-to-surface reduction — see §1 for the pitfall relative to Boone's per-window `TempDiff`).
3. **§3** Layer A — cooling rates × seafloor age (global orthographic centred on North America).
4. **§4** Layer B — cooling rates × paleotopography (North American regional zoom).
5. **§5** Layer C — cooling rates × Young 2022 `gld428` dynamic-topography uplift rate.
6. **§6** Multi-snapshot orthographic panel: `gld428` uplift rate + NA cooling samples through the Phanerozoic.
7. **§7** Cross-variable correlations — cooling rate vs `gld428` uplift rate, time-series, W vs E split at the Rocky Mountain front.

**Audience**: postgrad → researcher.
**Difficulty**: ★★★.
**Runtime**: ~4 min for a single snapshot; ~15 min for the full Phanerozoic time-series sweep at 20-Myr cadence.

## Data availability

**Hillenbrand, I.W., Thomson, K.D., Morgan, L.E., Gilmer, A.K., Engle, Z.T., Miller, A.T., Slawson, J., Dombrowski, A., Warrell, K.F., Malone, J., Souders, A.K., Hudson, A.M., Cosca, M.A., Paces, J.B., Thompson, R.A., and Park, A.J. (2023, rev. 5 June 2025)** — *USGS Geochron: A Database of Geochronological and Thermochronological Dates and Data (ver. 4.0)*. U.S. Geological Survey data release. DOI: <https://doi.org/10.5066/P9RZNPIF>. Landing: <https://www.sciencebase.gov/catalog/item/63345053d34e900e86c69f1a>.

License: **U.S. Government work — public domain (CC0-equivalent)**. No use constraints.

### Bundled subset

This notebook expects a CSV at `data/thermochronology_north_america/na_thermal_histories.csv` with the same column convention as the ThermoPlates Central Asia compilation:

```
lat, lon, sample_name, TOAGE, FROMAGE, OnlyCooling, TempDiff, Region
```

where `TempDiff` (°C/Myr) is the per-sample cooling rate — but computed here as the **closure-to-surface** average rate `(T_closure - T_surface) / closure_age`, NOT Boone's per-window inverse-t-T-model rate. USGS Geochron v4.0 publishes closure ages only (no packaged QTQt/HeFTy inverse t-T histories), so a single row is emitted per sample rather than the per-window rows Boone produces. This is a **coarser** reduction than the Boone convention; the trade-off is that the workflow works at continent-scale on any dataset publishing closure ages, without requiring published inverse thermal histories.

The `build_na_csv_from_usgs_geochron()` helper in §1 does the reduction: it joins `fissiontrack_USGS_data_release_v4.csv` + `uthhe_USGS_data_release_v4.csv` to `samples_USGS_data_release_v4.csv` on the sample-ID key, restricts to the North American lat/lon bounding box, filters by the user-configurable `MAX_SAMPLE_AGE_MA`, assigns per-method closure temperatures (AFT ~110°C, AHe ~70°C, ZHe ~180°C, ZFT ~230°C), and computes a single-window closure-to-surface cooling rate.

### To populate the data dir on first run

1. Visit <https://www.sciencebase.gov/catalog/item/63345053d34e900e86c69f1a>.
2. Download the ZIP release (15 CSVs + a data dictionary).
3. Extract to `data/thermochronology_north_america/usgs_geochron_v4/` — the CSV files should be at the top level of that directory.
4. Run cell 6 (§1). The helper reads the raw CSVs on first run and writes the flat `na_thermal_histories.csv` in the parent dir; subsequent runs skip the build step and read the flat CSV directly.

Until either the raw ZIP or the flat CSV is in place, this notebook fails fast on cell 1 of §1 with a clear `FileNotFoundError` naming both expected paths.


### Dynamic-topography data — Young et al. (2022) `gld428` 20-Myr cadence

The `gld428` plate-frame grids driving §5-§7 are **not bundled with the tutorial suite** (~810 MB across 51 NetCDF files spanning 0-980 Ma at 20-Myr cadence). The canonical download source is the EarthByte webdav: <https://www.earthbyte.org/webdav/ftp/Dynamic_Topography/gld428_m21/>. Once a Zenodo mirror is minted the DOI will be added here as an alternative source. Download the 51 NetCDF files and extract them to `data/Young2022_gld428_grids_20Myr/`.

Filename pattern: `gld428_PlateFrameGrid<age>.nc` where `<age>` is an integer (0, 20, 40, …, 980 Ma). Each grid has variable `z` on a regular lat/lon grid (2881 × 1441 = 0.125°). Provenance: Young et al. (2022, *EPSL* 584, 117451) — the same `gld428` mantle-flow simulation used by T22 (`gld428` sea-level change) and the removed T51 (Afro-Arabia). Unlike T22's `gld428` 5-Myr subset (which covers only 0-410 Ma), the 20-Myr subset covers the full Phanerozoic and into the Neoproterozoic.

Until the grids are in place, cell 16 (§5) fails fast with a clear `FileNotFoundError` naming the expected directory and the EarthByte source URL.

## Source

- Upstream data: Hillenbrand et al. 2023 USGS Geochron v4.0 — DOI 10.5066/P9RZNPIF.
- Sibling notebook for the methodology: T47 (Central Asia ThermoPlates workflow).

## Environment + imports

In [1]:
from pathlib import Path
import os, sys, warnings
if Path("../data").exists() and not Path("data").exists():
    os.chdir("..")

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

import gplately
import pygmt
from plate_model_manager import PlateModelManager

print("Environment")
print(f"  python      {sys.version.split()[0]}")
for _m in (np, pd, xr, gplately, pygmt):
    print(f"  {_m.__name__:11s} {getattr(_m, '__version__', 'n/a')}")

Environment
  python      3.12.5
  numpy       2.3.2
  pandas      2.2.3
  xarray      2026.4.0
  gplately    2.0.0.post19+git.2cce7bb3
  pygmt       v0.18.0


In [2]:
# === USER CONFIGURATION =====================================================
# Plate model + reference frame for §1-§4 (sample display / basemap context).
# Z22 mantle frame matches T47 / T48-T50 (no paleoclimate-vs-paleomag comparison
# is being drawn here). §5-§7 use a SEPARATE M21 NNR stack because that's the
# frame Young et al. (2022) ran gld428 in — see §5 for the plate-model-match
# rationale (CLAUDE.md "Plate-model-match rule").
MODEL_NAME             = "Zahirovic2022"
ANCHOR_PLATE_ID        = 0

# Headline snapshot age (Ma). 100 Ma is the Late Cretaceous — captures the
# North American Cordilleran story at its peak: active Sevier fold-and-thrust
# belt, Western Interior Seaway, and the very beginning of the Laramide
# flat-slab uplift phase (~80 Ma onward).
RECONSTRUCTION_TIME    = 100.0

# Filter USGS Geochron samples to closure ages ≤ this cap (Ma). Young 2022
# gld428 (20-Myr cadence) covers 0-980 Ma, so we can go all the way to the
# Cambrian; the default 540 Ma covers the full Phanerozoic which is where
# the geodynamic story is best-constrained. Cratonic samples with older
# closure ages (Neoproterozoic Canadian Shield etc.) exist in USGS Geochron
# v4 — raise the cap if you want to explore that regime.
MAX_SAMPLE_AGE_MA      = 540.0

# Inputs.
USGS_GEOCHRON_DIR      = Path("data/thermochronology_north_america/usgs_geochron_v4")
HISTORIES_CSV          = Path("data/thermochronology_north_america/na_thermal_histories.csv")
# PaleoDEM directory — geochemistry-corrected Scotese & Wright 2018 (Zhou,
# Müller, Farahbakhsh, in review). The bundled subset carries 16 ages
# (25-Myr cadence in the Phanerozoic).
PALEODEM_DIR_CANDIDATES = [
    Path("data/paleotopo_scotese/corrected_SW"),
    Path("external/Paleotopo_data_assimilation/data/corrected_Scotese"),
    Path.home() / "Documents/Software/Paleotopo_data_assimilation/data/corrected_Scotese",
]
PALEODEM_DIR           = next((p for p in PALEODEM_DIR_CANDIDATES if p.exists()),
                              PALEODEM_DIR_CANDIDATES[0])
print(f"  PaleoDEM dir resolved to: {PALEODEM_DIR} "
      f"({len(list(PALEODEM_DIR.glob('*_corrected_SW.nc')))} corrected_SW NCs available)")

# Cooling-rate classification thresholds (°C/Myr) — same as Boone's Central Asia.
FAST_CR                = 0.5      # `fast`
VERY_FAST_CR           = 1.0      # `very fast`
EXTRA_FAST_CR          = 1.5      # `extra fast`

# Cooling-rate colour-scale range — viridis 0-3 °C/Myr (matches T48/T49/T50).
COOLING_CPT_RANGE      = (0.0, 3.0)

# Region for the North American regional zoom + orthographic centre.
REGION_NORTH_AMERICA   = [-170, -50, 15, 75]      # [W, E, S, N] — Alaska to Mexico
GLOBAL_PROJ            = "G-100/40/16c"            # orthographic centred 100°W 40°N (roughly Kansas)
REGIONAL_PROJ          = "B-100/40/25/60/18c"      # Albers conic, centre 100°W 40°N, secants 25°N/60°N

# Subduction-kinematics tessellation density (radians along trench).
# 0.005 rad ~ 32 km segments — used only if kinematic overlays are added.
TESS_THRESHOLD_RAD     = 0.005

# Age sweep for the §7 cross-variable time-series. Full Phanerozoic in
# 20-Myr steps (matches gld428 cadence).
AGE_MIN_MA             = 0
AGE_MAX_MA             = 540
AGE_STEP_MA            = 20
AGES_MA                = list(range(AGE_MIN_MA, AGE_MAX_MA + 1, AGE_STEP_MA))
# ============================================================================
print(f"  time-series will step through {len(AGES_MA)} snapshot ages: "
      f"{AGES_MA[0]} → {AGES_MA[-1]} Ma, step {AGE_STEP_MA} Myr")

  PaleoDEM dir resolved to: data/paleotopo_scotese/corrected_SW (16 corrected_SW NCs available)
  time-series will step through 28 snapshot ages: 0 → 540 Ma, step 20 Myr


## 1. Load plate model + USGS Geochron v4.0 North American thermochronology

In [3]:
pmm = PlateModelManager()
model = pmm.get_model(MODEL_NAME, data_dir="./gplately_data")
recon = gplately.PlateReconstruction(
    rotation_model=model.get_rotation_model(),
    topology_features=model.get_topologies(),
    static_polygons=model.get_static_polygons(),
    anchor_plate_id=ANCHOR_PLATE_ID,
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=recon,
    coastlines=model.get_coastlines(),
    continents=model.get_continental_polygons(),
    COBs=model.get_COBs(),
    time=float(RECONSTRUCTION_TIME),
    plot_engine=gplately.PygmtPlotEngine(),
)


# ---------------------------------------------------------------------------
# Per-method closure temperatures (°C). Standard low-T thermochronology
# nominal values — refine per-sample if the source paper reports a
# specific effective closure temperature.
# ---------------------------------------------------------------------------
CLOSURE_T = {
    "AHe": 70.0,     # (U-Th)/He on apatite
    "AFT": 110.0,    # apatite fission track
    "ZHe": 180.0,    # (U-Th)/He on zircon
    "ZFT": 230.0,    # zircon fission track
}
SURFACE_T = 15.0     # °C — annual mean at mid-latitudes; fine at this precision
# The closure-to-surface reduction (single-window cooling rate) is physically
# meaningful only when the closure age is old enough for the cooling path to
# have been fully expressed. For very young ages, division by a tiny denominator
# produces enormous non-physical cooling rates; filter and cap to keep the
# signal Boone-scale (0.1–10 °C/Myr typical).
MIN_AGE_FOR_REDUCTION_MA = 1.0        # skip closure ages < this (Ma)
MAX_TEMPDIFF_CAP_CDEGMYR = 20.0       # cap cooling rate at this (°C/Myr)


def build_na_csv_from_usgs_geochron(usgs_dir: Path, out_csv: Path,
                                     max_sample_age_ma: float = 540.0,
                                     na_bbox=(-170, -50, 15, 75)) -> int:
    """Flatten USGS Geochron v4.0 into T47-style per-sample thermal-history
    rows, restricted to North America + samples with closure age
    ≤ max_sample_age_ma.

    Reads fissiontrack_USGS_data_release_v4.csv + uthhe_USGS_data_release_v4.csv,
    joins to samples_USGS_data_release_v4.csv on the sample-ID key,
    assigns a nominal closure temperature per (technique, mineral), and
    emits one row per sample with TempDiff = (T_closure - T_surface)
    / closure_age.

    Column-name notes for USGS Geochron v4:
      • samples CSV uses `lat` / `long` (not latitude/longitude).
      • technique CSVs use `samplematerial` = "apatite" | "zircon" | ...
        and `age` (Ma). The technique itself is implicit in which CSV
        the row came from (fissiontrack.csv → FT, uthhe.csv → He).
      • `country` = "United States" | "Canada" | "Mexico" | ..., and
        `stateprovince` names the state/province (or "not specified").

    Returns the number of rows written."""
    west, east, south, north = na_bbox
    samples_path = usgs_dir / "samples_USGS_data_release_v4.csv"
    ft_path      = usgs_dir / "fissiontrack_USGS_data_release_v4.csv"
    he_path      = usgs_dir / "uthhe_USGS_data_release_v4.csv"
    for p in (samples_path, ft_path, he_path):
        if not p.exists():
            raise FileNotFoundError(f"missing USGS Geochron file: {p}")

    # Load samples — only the columns we need (samples CSV is 33 MB)
    samples = pd.read_csv(samples_path,
                          usecols=["sampleid", "lat", "long",
                                   "country", "stateprovince"],
                          low_memory=False)
    samples["lat"]  = pd.to_numeric(samples["lat"],  errors="coerce")
    samples["long"] = pd.to_numeric(samples["long"], errors="coerce")
    samples = samples.dropna(subset=["sampleid", "lat", "long"])
    # NA bbox filter
    m = ((samples["long"] >= west) & (samples["long"] <= east)
         & (samples["lat"] >= south) & (samples["lat"] <= north))
    samples_na = samples.loc[m].reset_index(drop=True)
    print(f"    {len(samples_na):,} samples in NA bbox of {len(samples):,} total")

    def _mineral_to_method(samplematerial: str, tech: str) -> str:
        """Map (samplematerial, technique) to one of AHe / AFT / ZHe / ZFT."""
        m = str(samplematerial).strip().lower()
        if tech == "ft":
            if "apatite" in m: return "AFT"
            if "zircon"  in m: return "ZFT"
        elif tech == "he":
            if "apatite" in m: return "AHe"
            if "zircon"  in m: return "ZHe"
        return ""

    rows = []
    for path, tech in [(ft_path, "ft"), (he_path, "he")]:
        df = pd.read_csv(path, usecols=["sampleid", "samplematerial", "age",
                                         "ageintpnclass"],
                         low_memory=False)
        # Restrict to rows classified as cooling ages (the ageintpnclass column
        # separates crystallisation ages, mixed ages, cooling ages, etc.)
        if "ageintpnclass" in df.columns:
            df = df[df["ageintpnclass"].astype(str).str.lower().str.contains(
                    "cooling", na=False)].copy()
        df["age"] = pd.to_numeric(df["age"], errors="coerce")
        df = df.dropna(subset=["sampleid", "samplematerial", "age"])
        df = df[(df["age"] >= MIN_AGE_FOR_REDUCTION_MA)
                & (df["age"] <= max_sample_age_ma)]
        df["_method"] = df["samplematerial"].map(
            lambda s: _mineral_to_method(s, tech))
        df = df[df["_method"] != ""].copy()
        # Join to NA-filtered samples
        df = df.merge(samples_na, on="sampleid", how="inner")
        for _, r in df.iterrows():
            Tc = CLOSURE_T[r["_method"]]
            age = float(r["age"])
            temp_diff = min((Tc - SURFACE_T) / age, MAX_TEMPDIFF_CAP_CDEGMYR)
            rows.append({
                "lat": float(r["lat"]),
                "lon": float(r["long"]),
                "sample_name": f"{r['sampleid']}_{r['_method']}",
                "TOAGE": 0.0,
                "FROMAGE": age,
                "OnlyCooling": Tc - SURFACE_T,
                "TempDiff": temp_diff,
                "Region": str(r.get("stateprovince") or r.get("country") or "NA"),
                "Method": r["_method"],
            })

    if not rows:
        raise RuntimeError("USGS Geochron v4: no NA + low-T thermochron rows survived filters")

    out = pd.DataFrame(rows)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(out_csv, index=False)
    print(f"    wrote {len(out):,} sample rows → {out_csv}")
    return len(out)


# --- Build the flat CSV on first run, then load it ---
if not HISTORIES_CSV.exists():
    print(f"  ↻ {HISTORIES_CSV} not found — building from raw USGS Geochron CSVs …")
    build_na_csv_from_usgs_geochron(USGS_GEOCHRON_DIR, HISTORIES_CSV,
                                     max_sample_age_ma=MAX_SAMPLE_AGE_MA,
                                     na_bbox=tuple(REGION_NORTH_AMERICA))

hist = pd.read_csv(HISTORIES_CSV)
print(f"\n  loaded {len(hist):,} sample rows across "
      f"{hist['sample_name'].nunique():,} unique samples")
print(f"  methods present: "
      f"{hist['Method'].value_counts().to_dict() if 'Method' in hist.columns else '(no Method column)'}")
print(f"  closure-age range: {hist['FROMAGE'].min():.1f} – {hist['FROMAGE'].max():.1f} Ma")
print(f"  cooling-rate range: {hist['TempDiff'].min():.2f} – {hist['TempDiff'].max():.2f} °C/Myr")

  ↻ data/thermochronology_north_america/na_thermal_histories.csv not found — building from raw USGS Geochron CSVs …


FileNotFoundError: missing USGS Geochron file: data/thermochronology_north_america/usgs_geochron_v4/samples_USGS_data_release_v4.csv

## 2. Per-sample cooling rate at the snapshot age

In the USGS Geochron v4.0 reduction, each sample contributes a single row spanning the closure-to-surface interval (age 0 to closure age). At a given snapshot age `t`, a sample is "actively cooling" if `t` falls within its closure-to-surface window — i.e. `0 < t < closure_age`. The per-sample cooling rate returned is the flat (closure-to-surface average) rate — **coarser than Boone's per-window `TempDiff`** (see §1 caveat), but the workflow structure is otherwise identical.

In [ ]:
def per_sample_cooling_at(age_ma: float, hist_df: pd.DataFrame) -> pd.DataFrame:
    """One row per sample whose closure-to-surface window brackets age_ma.

    Because USGS Geochron v4 emits one row per sample (closure-to-surface),
    the selection is `TOAGE < age_ma < FROMAGE` — samples that were still
    cooling through their closure isotherm at age_ma."""
    snap = hist_df.loc[(hist_df["TOAGE"] < age_ma) & (age_ma < hist_df["FROMAGE"])].copy()
    snap["cooling_rate"] = pd.to_numeric(snap["TempDiff"], errors="coerce")
    snap = snap.dropna(subset=["cooling_rate", "lat", "lon"]).copy()
    snap["cooling_rate"] = snap["cooling_rate"].clip(lower=0)
    return (snap.groupby("sample_name", as_index=False)
                .agg(lat=("lat", "first"), lon=("lon", "first"),
                     cooling_rate=("cooling_rate", "mean")))

samples_now = per_sample_cooling_at(RECONSTRUCTION_TIME, hist)
print(f"  {len(samples_now)} samples bracket {RECONSTRUCTION_TIME:.0f} Ma; "
      f"cooling-rate range {samples_now.cooling_rate.min():.2f} – "
      f"{samples_now.cooling_rate.max():.2f} °C/Myr")

# Reconstruct samples to paleo-position (Z22 mantle frame)
gpts = gplately.Points(recon,
                        samples_now["lon"].to_numpy(float),
                        samples_now["lat"].to_numpy(float),
                        anchor_plate_id=ANCHOR_PLATE_ID)
rlons, rlats = gpts.reconstruct(RECONSTRUCTION_TIME,
                                 return_array=True,
                                 anchor_plate_id=ANCHOR_PLATE_ID)
samples_now = samples_now.assign(rlon=rlons, rlat=rlats).dropna(subset=["rlon","rlat"]).copy()
print(f"  reconstructed {len(samples_now)} samples to {RECONSTRUCTION_TIME:.0f} Ma")

## 3. Layer A — cooling rates × seafloor age (global orthographic centred on North America)

Same pattern as T48 §5: Z22 seafloor age grid as the basemap, thin-black plate-boundary backbone + styled ridges/trenches/teeth on top, then cooling-rate dots. The orthographic view centred on (100°W, 40°N) frames the North American sampling area together with its tectonic neighbours: the active Farallon (later Pacific-Cocos-Nazca) subduction margin to the west, the passive Atlantic margin to the east, and the Arctic margin to the north.

In [ ]:
# Z22 age grid (fetched on first run via PMM)
agegrid_da = xr.open_dataarray(
    model.get_raster("AgeGrids", time=float(RECONSTRUCTION_TIME))
)

fig_A = pygmt.Figure()
fig_A.basemap(region="d", projection=GLOBAL_PROJ, frame=["af"])
pygmt.makecpt(cmap="data/age_2020.cpt", series=[0, 250, 1], background="o")
fig_A.grdimage(agegrid_da, cmap=True)
gplot.time = float(RECONSTRUCTION_TIME)
gplot.plot_continents(fig_A, fill="gray95", pen="0.3p,gray40")
gplot.plot_all_topological_sections(fig_A, pen="0.5p,black")    # thin BLACK backbone
gplot.plot_ridges(fig_A, pen="0.8p,red3")
gplot.plot_trenches(fig_A, pen="0.8p,black")
gplot.plot_subduction_teeth(fig_A, color="black")
fig_A.colorbar(cmap=True, position="JBC+w8c/0.35c+h+o-5c/1c",
               frame=["xa50f10+lSeafloor age (Ma)"])
# Cooling-rate dots
pygmt.makecpt(cmap="viridis", series=[*COOLING_CPT_RANGE, 0.1], background="o")
fig_A.plot(x=samples_now["rlon"], y=samples_now["rlat"],
           fill=samples_now["cooling_rate"], cmap=True,
           style="c0.16c", pen="0.3p,black")
fig_A.colorbar(cmap=True, position="JBC+w8c/0.35c+h+o5c/1c",
               frame=["xa0.5f0.1+lCooling rate (@.C/Myr)"])
fig_A.text(text=f"{RECONSTRUCTION_TIME:.0f} Ma  ({MODEL_NAME}) — NA cooling × seafloor age",
           position="TL", offset="0.25c/-0.25c", justify="TL",
           font="12p,Helvetica-Bold,black", fill="white", pen="0.6p,gray40")
fig_A.show(width=900)

### What this figure shows you

At 100 Ma the North American plate is bracketed by an active Farallon subduction margin to the west (visible as the trench arc with subduction teeth along ~130-100°W), a young Atlantic Ocean opening to the east (with Cretaceous seafloor age visible offshore), and passive Arctic margins to the north. The **Sevier fold-and-thrust belt** was actively building along the western interior at this time, and the **Western Interior Seaway** flooded the mid-continent between the Cordillera and the Appalachian remnants.

Yellow viridis dots — fast-cooling samples — should concentrate along the **western Cordillera and the Sevier hinterland**, tracking the exhumation of the retro-arc mountain belt driven by Farallon flat-slab subduction and thick-skinned Laramide deformation that ramped up from ~80 Ma onward. Cratonic-interior samples (Great Plains, Canadian Shield) show slow cooling — as expected for a stable interior far from the active margin.

## 4. Layer B — cooling rates × paleotopography (North American regional zoom)

Albers conic zoom centred over North America. Geochemistry-corrected Scotese & Wright (2018) paleo-DEM (reused from T33) provides the topographic context — where inherited relief was at the snapshot age. Cooling-rate dots are expected to track regions of pre-existing high relief (Sevier fold-and-thrust belt, incipient Laramide arches, the Cordilleran interior) where reactivation drives cooling pulses.

In [ ]:
# Paleotopography (nearest 25-Myr step of corrected Scotese & Wright 2018)
# IMPORTANT: the corrected SW paleo-DEM is in Scotese & Wright's own plate
# frame. To overlay reconstructed samples honestly, we must reconstruct
# them with the SAME plate model that produced the DEM — NOT Z22. Mixing
# a DEM from one model with samples reconstructed via another model
# places the dots in a different absolute-position frame from the
# underlying grid (cf. CLAUDE.md: "Plate-model-match rule").
import re as _re
paleotopo_steps = sorted(
    int(_re.search(r"^(\d+)Ma_", p.name).group(1))
    for p in PALEODEM_DIR.glob("*Ma_corrected_SW.nc")
)
pt_age = min(paleotopo_steps, key=lambda a: abs(a - RECONSTRUCTION_TIME))
pt_path = PALEODEM_DIR / f"{pt_age}Ma_corrected_SW.nc"
pt_ds = xr.open_dataset(pt_path)
pt_da = pt_ds["M_corrected"] if "M_corrected" in pt_ds.data_vars else pt_ds["z"]
pt_land = pt_da.where(pt_da >= 0)
print(f"  paleotopo grid: {pt_path.name} ({pt_age} Ma — nearest to {RECONSTRUCTION_TIME:.0f} Ma)")

# Set up a Scotese & Wright reconstruction stack (separate from the Z22
# stack). It's defined in its own paleomag spin-axis frame at anchor 0.
model_sw = pmm.get_model("scotese_and_wright2018", data_dir="./gplately_data")
recon_sw = gplately.PlateReconstruction(
    rotation_model=model_sw.get_rotation_model(),
    topology_features=model_sw.get_topologies() if hasattr(model_sw, "get_topologies") else None,
    static_polygons=model_sw.get_static_polygons(),
    anchor_plate_id=0,
)
try:
    coastlines_sw = model_sw.get_coastlines()
except Exception:
    coastlines_sw = None
gplot_sw = gplately.PlotTopologies(
    plate_reconstruction=recon_sw,
    coastlines=coastlines_sw,
    continents=model_sw.get_continental_polygons(),
    COBs=None,
    time=float(pt_age),
    plot_engine=gplately.PygmtPlotEngine(),
)

# Snap the panel to the SW DEM's age — recompute cooling rate + reconstruct
# through SW at pt_age (same self-consistency pattern as T51's Layer B).
samples_sw_base = per_sample_cooling_at(pt_age, hist)
print(f"  cooling-rate snapshot at pt_age={pt_age} Ma: {len(samples_sw_base)} samples")

# Per-sample SW reconstruction (batched Points.reconstruct silently drops
# unreconstructable rows, so we loop and preserve NaNs)
_sw_rlon, _sw_rlat = [], []
for _, row in samples_sw_base.iterrows():
    try:
        gp = gplately.Points(recon_sw, [row["lon"]], [row["lat"]], anchor_plate_id=0)
        rl, ra = gp.reconstruct(pt_age, return_array=True, anchor_plate_id=0)
        _sw_rlon.append(float(rl[0]) if len(rl) else np.nan)
        _sw_rlat.append(float(ra[0]) if len(ra) else np.nan)
    except Exception:
        _sw_rlon.append(np.nan); _sw_rlat.append(np.nan)
samples_sw = samples_sw_base.assign(rlon_sw=_sw_rlon, rlat_sw=_sw_rlat)
samples_sw = samples_sw.dropna(subset=["rlon_sw", "rlat_sw"]).copy()
print(f"  reconstructed {len(samples_sw)} samples through SW to {pt_age} Ma")

fig_B = pygmt.Figure()
fig_B.basemap(region=REGION_NORTH_AMERICA, projection=REGIONAL_PROJ, frame=["af"])
pygmt.makecpt(cmap="gebco", series=[0, 4500, 250], background="o", continuous=True)
fig_B.grdimage(grid=pt_land, cmap=True, nan_transparent=True)
try:
    fig_B.plot(data=gplot_sw.get_continents(), pen="0.3p,gray40")
except Exception:
    pass
fig_B.colorbar(cmap=True, position="JBC+w8c/0.35c+h+o-5c/1c",
               frame=["xa1000f250+lPaleo-elevation (m, corrected SW)"])
pygmt.makecpt(cmap="viridis", series=[*COOLING_CPT_RANGE, 0.1], background="o")
fig_B.plot(x=samples_sw["rlon_sw"], y=samples_sw["rlat_sw"],
           fill=samples_sw["cooling_rate"], cmap=True,
           style="c0.20c", pen="0.4p,black")
fig_B.colorbar(cmap=True, position="JBC+w8c/0.35c+h+o5c/1c",
               frame=["xa0.5f0.1+lCooling rate (@.C/Myr)"])
fig_B.text(text=f"{pt_age} Ma  (SW plate frame) — NA cooling × paleotopography",
           position="TL", offset="0.25c/-0.25c", justify="TL",
           font="12p,Helvetica-Bold,black", fill="white", pen="0.6p,gray40")
fig_B.show(width=900)

### What this figure shows you

Layer B is rendered at the SW PaleoDEM's nearest cadence step to the configured snapshot age. The whole panel — sample cooling-rate snapshot, sample reconstruction, and DEM — sits in `scotese_and_wright2018`'s plate frame at that age (no Z22 quantities appear, respecting the plate-model-match rule).

What you see: the corrected Scotese & Wright paleo-DEM shows the inherited North American relief — the **Sevier fold-and-thrust belt** on the western side of the continent, remnants of the **Appalachian topography** on the east, and the low-relief cratonic interior in between. The fastest-cooling samples (bright viridis) should be located along the **flanks of the Sevier hinterland** and any earlier Laramide arches beginning to emerge — consistent with retro-arc exhumation of the Cordilleran mountain belt.

## 5. Cooling rates × dynamic-topography uplift rate — the Farallon-slab signal

For North America, **dynamic topography is a first-order driver** of Mesozoic-Cenozoic uplift, and Young et al. (2022) `gld428` — which explicitly captures subduction-coupled long-wavelength mantle flow — is the right substrate to test that hypothesis. Multiple independent lines of evidence link the shallow Farallon flat slab, its subsequent detachment (~40-25 Ma), and the associated mantle upwelling to the observed uplift signatures across the Colorado Plateau, the southern Rocky Mountains, and the western Great Plains (Moucha et al. 2008; Forte et al. 2010; Karlstrom et al. 2012; Liu et al. 2011).

Unlike the AARS case (where dynamic topography was a subordinate hypothesis relative to plume-driven uplift), the North American story sits directly in `gld428`'s wheelhouse — the model represents subduction-coupled DT well, and the **plume-suppression caveat is much less limiting** here.

We use the **plate-frame** `gld428` grids at 20-Myr cadence (0-980 Ma coverage) and compute the DT uplift rate as

$$\dot{z}_{\text{dyntopo}}(t) = \frac{DT_{\text{plate}}(t) - DT_{\text{plate}}(t + 20\,\text{Myr})}{20\,\text{Myr}}\quad [\text{m/Myr}]$$

Positive values indicate the mantle was lifting the surface over the preceding 20 Myr; negative values indicate dynamic subsidence. **The differencing is done in the plate frame** — where a point on the plate has a fixed (lat, lon) regardless of time — and the resulting Δz field is then rotated to the M21 NNR mantle frame at age `t` for paleo-geographic display. Differencing mantle-frame grids is forbidden by the suite conventions (see CLAUDE.md).

### Coverage and caveats

- The 20-Myr-cadence `gld428` grids are **not bundled with the tutorial suite** (~810 MB). Canonical download source: <https://www.earthbyte.org/webdav/ftp/Dynamic_Topography/gld428_m21/> (EarthByte webdav; a Zenodo mirror will be added on final release). Download the 51 NetCDFs into `data/Young2022_gld428_grids_20Myr/`, then re-run this cell.
- The `gld428` 20-Myr-cadence dataset covers **0-980 Ma** (51 snapshots), so we can push all the way back through the Paleozoic and into the Neoproterozoic. The default `MAX_SAMPLE_AGE_MA = 540` restricts to the Phanerozoic where the geodynamic story is best-constrained.
- `gld428` still suppresses thermal plumes by design — any plume-driven uplift is not represented. For North America this affects (at most) the Yellowstone track post-16 Ma (a small subset of samples) and possibly some Neoproterozoic-Cambrian rifting events (long before Yellowstone). Most of the Farallon-slab-linked signal we care about is captured.
- The 20-Myr cadence is coarser than gmcm9's 5-Myr cadence but adequate for the long-wavelength signal we want to correlate against per-sample cooling rates.

In [ ]:
# === USER CONFIGURATION — §5/§6/§7 (dyntopo additions) ==================
# Snapshot ages for the multi-panel figure. Six well-spaced ages spanning
# the Phanerozoic → 2×3 panel grid. Adjust these to zoom into any
# geodynamic epoch of interest.
DYNTOPO_SNAPSHOT_AGES = [60, 100, 200, 300, 400, 500]
# Step (Myr) for the uplift-rate difference. Must match the gld428
# cadence in the bundled 20-Myr dataset.
DYNTOPO_STEP_MYR      = 20
# ±50 m/Myr cpt range for the uplift-rate basemap (matches T23).
UPLIFT_RANGE_M_PER_MYR = (-50, 50, 2)

# Plate model for sample reconstruction in §5/§6/§7. Young et al. (2022)
# ran gld428 in the M21 NNR (Merdith 2021 no-net-rotation) frame — so per
# the plate-model-match rule we set up a separate stack here. The Z22
# stack used in §1-§4 stays untouched.
M21NNR_DIR            = Path("data/young2022_dyntopo/M21NNR")

# Young 2022 gld428 PLATE-FRAME 20-Myr cadence grids. Filenames:
# "gld428_PlateFrameGrid<age>.nc" where <age> is an integer (0, 20, 40, ...).
GLD428_DIR_CANDIDATES = [
    Path("data/Young2022_gld428_grids_20Myr"),
    Path("external/Young2022_gld428_grids_20Myr"),
    Path.home() / "Documents/GPlates/GPlately-pyGMT_tutorials/data/Young2022_gld428_grids_20Myr",
]
GLD428_DIR = next((p for p in GLD428_DIR_CANDIDATES if p.exists()),
                  GLD428_DIR_CANDIDATES[0])
_gld_ncs = list(GLD428_DIR.glob("gld428_PlateFrameGrid*.nc")) if GLD428_DIR.exists() else []
if not _gld_ncs:
    raise FileNotFoundError(
        f"gld428 20-Myr grids not found at {GLD428_DIR}. "
        "Download the 51 gld428 NetCDFs (gld428_PlateFrameGrid<age>.nc, 0-980 Ma at 20-Myr "
        "cadence) from the EarthByte webdav: "
        "https://www.earthbyte.org/webdav/ftp/Dynamic_Topography/gld428_m21/  and extract "
        f"them into {GLD428_DIR}. See §1 Data availability for the full recipe."
    )
print(f"  gld428 PLATE-frame dir: {GLD428_DIR}")
print(f"    NCs available: {len(_gld_ncs)}")
print(f"  M21 NNR plate-model dir: {M21NNR_DIR}")

# === M21 NNR PlateReconstruction (separate from the Z22 stack in §1-§4) ===
import pygplates as _pgp
m21_rot_files  = sorted(M21NNR_DIR.glob("*.rot"))
m21_all_gpml   = sorted(M21NNR_DIR.glob("*.gpml"))
m21_topo_files = [f for f in m21_all_gpml
                  if "shapes_continents" not in f.name
                  and "shapes_cratons"   not in f.name]
m21_continents_fc = _pgp.FeatureCollection(str(M21NNR_DIR / "shapes_continents_Merdith_et_al.gpml"))
recon_m21 = gplately.PlateReconstruction(
    rotation_model=_pgp.RotationModel([str(p) for p in m21_rot_files]),
    topology_features=[_pgp.FeatureCollection(str(f)) for f in m21_topo_files],
    static_polygons=m21_continents_fc,
    anchor_plate_id=0,
)


# ---- gld428 plate-frame Δz helper ----------------------------------------
def _gld428_path(age_ma: float) -> Path:
    """Return the path to the gld428 grid at the nearest 20-Myr step."""
    a = int(round(age_ma / DYNTOPO_STEP_MYR) * DYNTOPO_STEP_MYR)
    return GLD428_DIR / f"gld428_PlateFrameGrid{a}.nc"


def uplift_rate_for_display(age_ma: float) -> xr.DataArray:
    """Plate-frame Δz (m/Myr) rotated to the M21 NNR mantle frame at age_ma.
    Uses gld428 grids at age_ma and age_ma + DYNTOPO_STEP_MYR."""
    p0 = _gld428_path(age_ma)
    p1 = _gld428_path(age_ma + DYNTOPO_STEP_MYR)
    if not p0.exists() or not p1.exists():
        raise FileNotFoundError(f"missing gld428 pair: {p0.name} / {p1.name}")
    ds0 = xr.open_dataset(p0); ds1 = xr.open_dataset(p1)
    z0 = ds0["z"]; z1 = ds1["z"]
    dz_pf = (z0 - z1) / float(DYNTOPO_STEP_MYR)   # m/Myr, plate frame
    # Rotate plate-frame Δz to M21 NNR mantle frame at age_ma via
    # gplately.Raster.reconstruct() (T23 helper pattern).
    lon = dz_pf.lon.values; lat = dz_pf.lat.values
    r = gplately.Raster(data=dz_pf.values.astype(np.float32),
                        plate_reconstruction=recon_m21, time=0,
                        extent=[lon.min(), lon.max(), lat.min(), lat.max()])
    rr = r.reconstruct(time=float(age_ma), threads=1,
                       anchor_plate_id=0, inplace=False)
    return xr.DataArray(np.asarray(rr.data), dims=("lat", "lon"),
                        coords={"lat": lat, "lon": lon}, name="uplift_rate")


# ---- sample side: reconstruct through M21 NNR + sample gld428 at present-day
# lat/lon (plate-frame is fixed to the plate, so present-day lat/lon is the
# right sampling location — see T50 build_master convention) ------
def sample_dyntopo_at_age(age_ma: float, hist_df: pd.DataFrame) -> pd.DataFrame:
    """Per-sample cooling rate + gld428 dyntopo uplift rate at age_ma."""
    samp = per_sample_cooling_at(age_ma, hist_df)
    if samp.empty:
        return samp.assign(rlon_m21=[], rlat_m21=[], uplift_rate_m_per_Myr=[])
    # Reconstruct through M21 NNR (mantle frame) for display
    gp = gplately.Points(recon_m21,
                          samp["lon"].to_numpy(float),
                          samp["lat"].to_numpy(float),
                          anchor_plate_id=0)
    rl, ra = gp.reconstruct(age_ma, return_array=True, anchor_plate_id=0)
    samp = samp.assign(rlon_m21=rl, rlat_m21=ra)
    # Sample plate-frame gld428 uplift rate at each sample's PRESENT-DAY
    # lat/lon (plate frame is rigidly attached to the plate → present-day
    # coord is the right lookup)
    try:
        upl_pf = uplift_rate_for_display(float(age_ma))
        vals = upl_pf.interp(lon=xr.DataArray(samp["lon"].values, dims="s"),
                              lat=xr.DataArray(samp["lat"].values, dims="s"),
                              method="linear").values
        samp["uplift_rate_m_per_Myr"] = vals
    except FileNotFoundError:
        samp["uplift_rate_m_per_Myr"] = np.nan
    return samp

print("  ✓ gld428 + M21 NNR helpers set up")

### How to read the dyntopo basemap

The polar (red-blue) cpt is centred at **zero uplift rate**: red = surface lifted by mantle flow over the preceding 20 Myr (broad mantle upwelling underneath), blue = surface subsided (mantle downwelling, typically slab-pull from a nearby subduction zone). ±50 m/Myr saturates the strongest features. NA cooling-rate dots overlay this, coloured by per-sample rate.

For North America the signal we're chasing is the **long-wavelength Farallon-slab dynamic topography**:

- The Cordilleran margin should show **subsidence** (blue) during flat-slab periods (~90-40 Ma), as the shallowly-underthrust slab drags the base of the continental lithosphere downward.
- Following slab detachment (~40-25 Ma) and mantle re-influx into the vacated space, the western interior should transition to **uplift** (red), producing the widely-recognised Cenozoic exhumation phase across the Colorado Plateau + southern Rockies.
- The cratonic interior (Great Plains, Canadian Shield) should show subtle DT modulation, mostly slow.

If NA cooling rates correlate positively with `gld428` uplift rate through the Cretaceous-Cenozoic, that's direct evidence that the dominant driver of NA exhumation is mantle-flow uplift — i.e., the Farallon story.

## 6. Multi-snapshot panel: gld428 uplift rate + NA cooling samples through time

Six orthographic globes centred on North America, one per snapshot age in `DYNTOPO_SNAPSHOT_AGES`. The Farallon-slab-linked subsidence-then-uplift transition should be visible as the pattern evolves through the Cretaceous-Cenozoic. Earlier Paleozoic snapshots probe subduction-margin dynamic topography around the Iapetus/Rheic closures.

In [ ]:
fig_dt = pygmt.Figure()
n_ages = len(DYNTOPO_SNAPSHOT_AGES)
n_cols = 3     # 2×3 grid for 6 snapshot ages
n_rows = (n_ages + n_cols - 1) // n_cols

# NA-centred orthographic projection for every panel
DYNTOPO_PROJ = "G-100/40/8c"

with fig_dt.subplot(nrows=n_rows, ncols=n_cols, figsize=("28c", "20c"),
                    sharex="b", sharey="l", margins=["0.3c", "0.4c"]):
    for idx, age in enumerate(DYNTOPO_SNAPSHOT_AGES):
        with fig_dt.set_panel(panel=idx):
            try:
                upl_rate = uplift_rate_for_display(float(age))
                upl_rate.gmt.registration = 0; upl_rate.gmt.gtype = 1
                fig_dt.basemap(region="d", projection=DYNTOPO_PROJ, frame=["af"])
                pygmt.makecpt(cmap="polar", series=list(UPLIFT_RANGE_M_PER_MYR),
                               continuous=True, background="o")
                fig_dt.grdimage(grid=upl_rate, cmap=True, nan_transparent=True)
                # M21 NNR continental outlines at this age
                gplot_m = gplately.PlotTopologies(
                    plate_reconstruction=recon_m21,
                    coastlines=None, continents=m21_continents_fc, COBs=None,
                    time=float(age), plot_engine=gplately.PygmtPlotEngine())
                try:
                    fig_dt.plot(data=gplot_m.get_continents(), pen="0.4p,gray20")
                except Exception:
                    pass
                # Samples at this age — reconstructed via M21 NNR for display
                samp = sample_dyntopo_at_age(float(age), hist)
                if not samp.empty:
                    samp_plot = samp.dropna(subset=["rlon_m21", "rlat_m21"])
                    if not samp_plot.empty:
                        pygmt.makecpt(cmap="viridis", series=[*COOLING_CPT_RANGE, 0.1],
                                       background="o")
                        fig_dt.plot(x=samp_plot["rlon_m21"], y=samp_plot["rlat_m21"],
                                     fill=samp_plot["cooling_rate"], cmap=True,
                                     style="c0.18c", pen="0.3p,black")
                fig_dt.text(text=f"{int(age)} Ma  (n={len(samp)})",
                             position="TL", offset="0.2c/-0.2c", justify="TL",
                             font="11p,Helvetica-Bold,black",
                             fill="white", pen="0.4p,gray40")
            except FileNotFoundError as e:
                print(f"  ⚠ {age} Ma: {e}")

# Shared bottom colour bars (uplift rate + cooling rate)
pygmt.makecpt(cmap="polar", series=list(UPLIFT_RANGE_M_PER_MYR),
               continuous=True, background="o")
fig_dt.colorbar(cmap=True, position="JBC+w10c/0.35c+h+o-6c/1.4c",
                 frame=["xa20f5+lgld428 uplift rate (m/Myr)"])
pygmt.makecpt(cmap="viridis", series=[*COOLING_CPT_RANGE, 0.1], background="o")
fig_dt.colorbar(cmap=True, position="JBC+w10c/0.35c+h+o6c/1.4c",
                 frame=["xa0.5f0.1+lCooling rate (@.C/Myr)"])
fig_dt.show(width=1200)

### What these globes tell you

The six panels (60, 100, 200, 300, 400, 500 Ma) trace the long-wavelength dynamic-topography uplift rate at NA sample positions from the Cambrian through the Cenozoic. Because `gld428` explicitly captures subduction-coupled DT (see §5), the North American signal we care about should be **directly represented** in these maps:

- **60 Ma (Paleocene)**: the tail end of the Laramide flat-slab phase — the shallow Farallon slab drags the base of the Cordilleran interior downward, producing a broad blue (subsidence) patch across the western interior.
- **100 Ma (Cenomanian)**: mid-Cretaceous Farallon subduction has been ongoing for ~50 Myr; expect subsidence bands aligned with the active margin and moderate mantle-return-flow uplift patches over the continental interior.
- **200 Ma (Early Jurassic)**: near the onset of Farallon-family subduction (Cordilleran orogen inception); Pangea break-up is barely under way, so the DT signal is dominated by mantle warming under the still-thick supercontinent.
- **300 Ma (Late Carboniferous)**: Alleghanian orogeny active on the eastern margin as Pangea assembles; DT signal is dominated by the collisional suture zone.
- **400 Ma (Early Devonian)**: Acadian orogeny active, Iapetus recently closed; DT signal on the Appalachian side.
- **500 Ma (Cambrian)**: passive margin phase for both sides of Laurentia, minimal DT activity expected.

Compare the NA cooling-rate dots overlaid on each panel against these DT signatures. Persistent visual co-location of the fastest-cooling samples (yellow viridis) with red (uplift) patches would be evidence for the DT-driven-exhumation hypothesis; persistent co-location with blue (subsidence) patches or no consistent pattern would argue for competing drivers (climate, tectonic uplift decoupled from mantle flow, or DT signal not resolved at this cadence).

## 7. Cross-variable correlations — cooling rate vs DT uplift rate, time-series

Same pipeline as the earlier convergence-rate analysis, but with `gld428` uplift rate replacing the convergence rate. For each age slice with at least three NA samples actively cooling, we sample `gld428` at the M21 NNR-reconstructed sample positions and compute Pearson `r(cooling, uplift_rate)`. A positive trend across many ages would support the plume-driven-exhumation hypothesis.

In [ ]:
# Build per-age master tables (cooling rate + dyntopo uplift rate at M21 NNR pos)
DYNTOPO_MASTER_DIR = Path("data/thermochronology_north_america/dyntopo_master")
DYNTOPO_MASTER_DIR.mkdir(parents=True, exist_ok=True)

def dyntopo_master_path(age): return DYNTOPO_MASTER_DIR / f"dyntopo_master_{int(age):03d}Ma.parquet"

# Build for ages within gld428 coverage
DYNTOPO_AGES      = [a for a in AGES_MA if a + DYNTOPO_STEP_MYR <= 980]
# Age list for cooling-vs-uplift Pearson r sweep — use the same set here
# (no plume-suppression restriction is required for NA-Farallon; §5 argues
# gld428 captures the relevant subduction-coupled DT).
DYNTOPO_AGES_CORR = list(DYNTOPO_AGES)
n_built = n_cached = n_skipped = 0
for age in DYNTOPO_AGES:
    p = dyntopo_master_path(age)
    if p.exists(): n_cached += 1; continue
    try:
        df = sample_dyntopo_at_age(float(age), hist)
        if df is None or df.empty: n_skipped += 1; continue
        df["age_Ma"] = float(age)
        df.to_parquet(p); n_built += 1
    except FileNotFoundError:
        n_skipped += 1
print(f"  ✓ built {n_built} new parquet(s); {n_cached} cached; {n_skipped} skipped (no data)")

### Time-series of cooling rate AND dyntopo uplift rate — W vs E NA samples

Two stacked panels spanning the age range set by `AGES_MA`:

- **Top — cooling rate**: derived directly from USGS Geochron thermochronology and unaffected by any `gld428` model limitations.
- **Bottom — `gld428` uplift rate**: sampled at each sample's present-day position (plate-frame attached).

The samples split roughly at the Rocky Mountain front (~-100°E, aligning with the transition from Cordilleran deformation to cratonic interior):

- **Western sector (lon ≤ -100°E)** — Cordillera, Basin & Range, Colorado Plateau, Sierras, Cascades, Coast Ranges
- **Eastern sector (lon > -100°E)** — Great Plains, Midcontinent, Appalachians, Canadian Shield east of ~-100°E

Median ± inter-quartile-range bands per sector; faint background traces are individual sample trajectories. The Farallon story predicts the western sector should show high cooling rates + positive uplift rate in the Late-Cretaceous-to-Cenozoic window; the eastern sector should show slower cooling + weaker DT modulation.

In [ ]:
# Pool master parquet files + split W vs E at -100°E (Rocky Mountain front)
EW_THRESHOLD_LON = -100.0
frames = []
for a in DYNTOPO_AGES:
    p = dyntopo_master_path(a)
    if not p.exists(): continue
    df = pd.read_parquet(p)
    if "age_Ma" not in df.columns: df["age_Ma"] = float(a)
    frames.append(df)
ts_pool = pd.concat(frames, ignore_index=True)
ts_pool["sector"] = np.where(ts_pool["lon"] <= EW_THRESHOLD_LON, "W", "E")

def summarise(metric_col):
    return (ts_pool.dropna(subset=[metric_col])
                   .groupby(["age_Ma", "sector"])[metric_col]
                   .agg(median="median", q25=lambda s: s.quantile(0.25),
                        q75=lambda s: s.quantile(0.75), n="count").reset_index())
summ_cool = summarise("cooling_rate")
summ_upl  = summarise("uplift_rate_m_per_Myr")

# Colours: brown for the Cordilleran west, green for the cratonic east
sector_colors = {"W": "#8B4513", "E": "#2E7D32"}
sector_labels = {"W": "Western (lon ≤ -100°E)  — Cordillera, Basin & Range, Colorado Plateau",
                 "E": "Eastern (lon > -100°E)  — Great Plains, Midcontinent, Appalachians"}

fig, (ax_c, ax_u) = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

# --- top panel: COOLING RATE ---
for sname, g in ts_pool.dropna(subset=["cooling_rate"]).groupby("sample_name"):
    g = g.sort_values("age_Ma"); sec = g["sector"].iloc[0]
    ax_c.plot(g["age_Ma"], g["cooling_rate"], color=sector_colors[sec], alpha=0.03, lw=0.5)
for sec in ("W", "E"):
    sub = summ_cool[summ_cool["sector"] == sec].sort_values("age_Ma")
    ax_c.fill_between(sub["age_Ma"], sub["q25"], sub["q75"],
                       color=sector_colors[sec], alpha=0.18, label=f"{sector_labels[sec]}  (IQR)")
    ax_c.plot(sub["age_Ma"], sub["median"], color=sector_colors[sec],
              lw=2.2, marker="o", markersize=5, markerfacecolor="white",
              markeredgewidth=1.2, label=f"{sector_labels[sec]}  (median)")
ax_c.set_ylabel("Cooling rate (°C/Myr)")
ax_c.grid(alpha=0.3)
ax_c.set_title("NA cooling rate vs reconstruction age — W/E split at the Rocky Mountain front",
                loc="left")
ax_c.legend(loc="upper left", fontsize=9, framealpha=0.92, ncol=1)

# --- bottom panel: gld428 UPLIFT RATE ---
for sname, g in ts_pool.dropna(subset=["uplift_rate_m_per_Myr"]).groupby("sample_name"):
    g = g.sort_values("age_Ma"); sec = g["sector"].iloc[0]
    ax_u.plot(g["age_Ma"], g["uplift_rate_m_per_Myr"], color=sector_colors[sec], alpha=0.03, lw=0.5)
for sec in ("W", "E"):
    sub = summ_upl[summ_upl["sector"] == sec].sort_values("age_Ma")
    ax_u.fill_between(sub["age_Ma"], sub["q25"], sub["q75"],
                       color=sector_colors[sec], alpha=0.18, label=f"{sector_labels[sec]}  (IQR)")
    ax_u.plot(sub["age_Ma"], sub["median"], color=sector_colors[sec],
              lw=2.2, marker="o", markersize=5, markerfacecolor="white",
              markeredgewidth=1.2, label=f"{sector_labels[sec]}  (median)")
ax_u.axhline(0, color="black", lw=0.6, ls="--", alpha=0.5)
ax_u.set_xlabel("Reconstruction age (Ma)")
ax_u.set_ylabel("gld428 dyntopo uplift rate (m/Myr)")
ax_u.grid(alpha=0.3)
ax_u.legend(loc="upper left", fontsize=9, framealpha=0.92, ncol=1)
ax_u.invert_xaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Pool + Pearson r per age + scatter plot
frames = [pd.read_parquet(dyntopo_master_path(a))
          for a in DYNTOPO_AGES_CORR if dyntopo_master_path(a).exists()]
pool = pd.concat(frames, ignore_index=True)
pool = pool.dropna(subset=["uplift_rate_m_per_Myr"]).copy()
if "age_Ma" not in pool.columns:
    rows = []
    for a in DYNTOPO_AGES_CORR:
        p = dyntopo_master_path(a)
        if not p.exists(): continue
        df = pd.read_parquet(p); df["age_Ma"] = float(a); rows.append(df)
    pool = pd.concat(rows, ignore_index=True).dropna(subset=["uplift_rate_m_per_Myr"])
print(f"  pooled: {len(pool):,} (sample, age) pairs across "
      f"{pool['age_Ma'].nunique()} ages")


def pearson_r(x, y):
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3: return np.nan, int(m.sum())
    return float(np.corrcoef(x[m], y[m])[0, 1]), int(m.sum())


r_rows = []
for age, sub in pool.groupby("age_Ma"):
    r, n = pearson_r(sub["uplift_rate_m_per_Myr"].to_numpy(float),
                      sub["cooling_rate"].to_numpy(float))
    r_rows.append({"age_Ma": age, "r_cool_vs_uplift": r, "n": n})
r_by_age = pd.DataFrame(r_rows).sort_values("age_Ma")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8))
df = pool.dropna(subset=["uplift_rate_m_per_Myr", "cooling_rate"])
age_norm = Normalize(min(DYNTOPO_AGES_CORR), max(DYNTOPO_AGES_CORR))
sc = ax1.scatter(df["uplift_rate_m_per_Myr"], df["cooling_rate"],
                  c=df["age_Ma"], cmap="viridis", norm=age_norm,
                  s=18, edgecolor="black", linewidth=0.2, alpha=0.7)
if len(df) > 1:
    slope, intercept = np.polyfit(df["uplift_rate_m_per_Myr"], df["cooling_rate"], 1)
    xl = np.linspace(df["uplift_rate_m_per_Myr"].min(),
                     df["uplift_rate_m_per_Myr"].max(), 100)
    ax1.plot(xl, slope*xl + intercept, color="black", lw=1.4)
r_all, n_all = pearson_r(df["uplift_rate_m_per_Myr"].values, df["cooling_rate"].values)
ax1.text(0.04, 0.96, f"r = {r_all:+.3f}  (n = {n_all:,})",
         transform=ax1.transAxes, va="top", ha="left", fontsize=11,
         bbox=dict(facecolor="white", edgecolor="0.40", boxstyle="round,pad=0.3"))
ax1.axvline(0, color="black", lw=0.6, ls="--", alpha=0.5)
ax1.set_xlabel("gld428 dyntopo uplift rate (m/Myr) at M21 NNR-reconstructed sample position")
ax1.set_ylabel("NA cooling rate (°C/Myr)")
ax1.grid(alpha=0.3)
cbar = fig.colorbar(sc, ax=ax1, pad=0.02, aspect=30)
cbar.set_label("Reconstruction age (Ma)")

# Pearson r time-series
rmax = max(0.2, r_by_age["r_cool_vs_uplift"].abs().max() if not r_by_age.empty else 0.2)
ax2.axhspan(0, rmax, color="lightcoral", alpha=0.15)
ax2.axhspan(-rmax, 0, color="lightsteelblue", alpha=0.15)
ax2.axhline(0, color="black", lw=0.6, ls="--", alpha=0.5)
ax2.plot(r_by_age["age_Ma"], r_by_age["r_cool_vs_uplift"],
         color="black", lw=1.8, marker="o", markersize=6,
         markerfacecolor="white", markeredgewidth=1.4)
ax2.set_xlabel("Reconstruction age (Ma)")
ax2.set_ylabel("Pearson r  (cooling rate vs gld428 uplift rate)")
ax2.set_ylim(-rmax, rmax)
ax2.grid(alpha=0.3)
ax2.invert_xaxis()

plt.tight_layout()
plt.show()

### Interpreting the r(t) trajectory

For North America the strongest positive `r(t)` signal is expected in the **Late Cretaceous through Paleogene** window (~90-30 Ma) — the interval where Farallon flat-slab subduction and its detachment drove the largest DT variations across the Cordilleran interior. Persistent positive `r(t)` in that window would be direct statistical evidence that Cordilleran cooling rates were paced by mantle-flow-driven surface uplift — the textbook Farallon story.

**Contrast with the AARS case**: for Afro-Arabia (in the removed T51), we had to restrict `r(t)` to ≥50 Ma because `gld428` suppresses plumes and the Afar plume dominates the post-50 Ma African uplift history. For North America, no such restriction is required — the dominant post-100 Ma NA uplift driver (the Farallon slab and its subsequent detachment) is exactly what `gld428` DOES capture. This makes the NA case a **cleaner test** of the DT-thermochron coupling than the AARS case was.

**Important caveats**:
- The USGS Geochron closure-to-surface cooling-rate reduction is coarser than Boone's per-window `TempDiff`; a sample with a complex t-T history (multi-stage cooling, reheating events) is compressed to a single mean rate. This blurs the correlation compared with a Boone-style analysis on samples that have published inverse thermal-history models. For the ~5-10% of the compilation that DOES have inverse t-T models in the primary literature, an "Extend this" exercise could recompute per-window rates and rerun the correlation for that subset.
- `gld428` at 20-Myr cadence resolves long-wavelength DT well but under-represents rapid events like the Yellowstone-hotspot passage (post-16 Ma) — a plume-suppressed feature in this model.
- The plate-model match is respected by construction: samples reconstructed via M21 NNR (the frame `gld428` was run in), and Δz differencing done in the plate frame then rotated to the mantle frame for map display.
- Sample density is uneven — Cordilleran samples dominate USGS Geochron v4's low-T thermochronology coverage, so the E sector is under-represented and its statistics are noisier.

## Extend this

- **Per-window `TempDiff` on the subset with published inverse thermal-history models.** For samples whose primary literature includes QTQt or HeFTy inverse models (~5-10% of USGS Geochron entries), replace the closure-to-surface reduction in cell 6 with per-window rates read from the published models. The Cordillera Zenodo stack (Kaempfer 2024, Ronemus 2023, Schoettle-Greene 2022, Benowitz 2023, Colgan 2020, Lease 2021) provides the largest single body of open-license inverse t-T histories over the NA-Cordilleran window and would sharpen the DT correlation considerably.
- **Method-stratified correlation.** Restrict the pooled scatter (cell 25) to a single thermochronometer (AHe, AFT, ZHe, or ZFT) to see whether the DT-cooling correlation depends on closure temperature. A cleaner AHe signal would suggest the near-surface unroofing tracks DT most directly; a cleaner AFT signal would push the coupling deeper.
- **Fault-proximity test (T49 template).** The GEM Global Active Faults Database + Utah Geological Survey compilations cover NA well. Mirror T49's proximity-analysis block here: does the fastest-cooling subset preferentially lie within 25 km of active-Cordilleran + Basin-Range faults? Contrast with cratonic-interior samples.
- **Compare against a Farallon-slab reconstruction.** The Sigloch & Mihalynuk (2013) slab-strand reconstruction gives a direct 3-D geometry for the Farallon slab through the Mesozoic. Plot the sample positions against slab depth-slice contours at each snapshot age and test whether cooling rate correlates with vertical slab depth.
- **Extend to the Canadian Shield.** Raise `MAX_SAMPLE_AGE_MA` to 1000 Ma to include Neoproterozoic-aged cratonic samples. `gld428` extends to 980 Ma so the DT context is available; the question is whether AFT/He ages older than 540 Ma preserve any DT-linked signal or whether cratonic thermal histories are dominated by isostatic denudation.

## References

- Hillenbrand, I.W., et al. (2023, rev. 2025). *USGS Geochron: A Database of Geochronological and Thermochronological Dates and Data (ver. 4.0)*. U.S. Geological Survey data release. DOI: 10.5066/P9RZNPIF.
- Young, A., Flament, N., Williams, S.E., Merdith, A., Cao, X., Müller, R.D. (2022). Long-term Phanerozoic sea level change from solid Earth processes. *Earth and Planetary Science Letters* 584, 117451.
- Boone, S.C. et al. (2025). Phanerozoic thermochronology record of Afro-Arabia through space and time. *Scientific Data* 12, 472.
- Moucha, R., Forte, A.M., Rowley, D.B., Mitrovica, J.X., Simmons, N.A., Grand, S.P. (2008). Mantle convection and the recent evolution of the Colorado Plateau and the Rio Grande Rift Valley. *Geology* 36, 439-442.
- Forte, A.M., Moucha, R., Simmons, N.A., Grand, S.P., Mitrovica, J.X. (2010). Descent of the ancient Farallon slab drives localized mantle flow below the North American surface: evidence from time-varying kinematics of North America and its coastal ocean. *Geophysical Research Letters* 37, L18309.
- Karlstrom, K.E., Coblentz, D., Dueker, K., Ouimet, W., Kirby, E., Van Wijk, J., Schmandt, B., Kelley, S., Lazear, G., Crossey, L.J., et al. (2012). Mantle-driven dynamic uplift of the Rocky Mountains and Colorado Plateau and its surface response: toward a unified hypothesis. *Lithosphere* 4, 3-22.
- Liu, L., Gurnis, M., Seton, M., Saleeby, J., Müller, R.D., Jackson, J.M. (2011). The role of oceanic plateau subduction in the Laramide orogeny. *Nature Geoscience* 3, 353-357.
- Merdith, A.S., Williams, S.E., Collins, A.S., et al. (2021). Extending full-plate tectonic models into deep time. *Earth-Science Reviews* 214, 103477.
- Zahirovic, S., Eleish, A., Doss, S., Ma, X., Cannon, J., Müller, R.D., Fox, P. (2022). Active deforming-plate Phanerozoic reconstruction. *Earth-Science Reviews* 233, 104145.